In [1]:
import pandas as pd
import numpy as np
import warnings
from micom.qiime_formats import load_qiime_medium
from micom.workflows import grow, build
from micom.interaction import interactions
from micom.workflows import load_results, GrowthResults
from zipfile import ZipFile

warnings.filterwarnings('ignore')

## Preprocess normalized abundance dataframe to meet MICOM's requirement and generate metabolic models for each sample

In [2]:
reference_taxonomy = pd.read_csv('./processed/reference_taxonomy.csv').drop(columns=['Unnamed: 0'])
agora_genomes = pd.read_csv('./processed/genome_size_ungapped.csv').drop(columns=['Unnamed: 0'])
reference_taxonomy_counts = pd.merge(reference_taxonomy, agora_genomes, on='id', how='right')
reference_taxonomy_counts['NCBI Taxonomy ID'] = reference_taxonomy_counts['NCBI Taxonomy ID'].astype(int).astype(str)
abundance = pd.read_csv('./raw/goll_agora_map.counts.csv')
abundance['taxon_id'] = abundance['taxon_id'].astype(str)

cols_to_keep = ['taxon_id'] + [col for col in abundance.columns if col.startswith('D') or col.endswith('-0') or col.endswith('-6') ]
abundance = abundance[cols_to_keep]
abundance = abundance.merge(reference_taxonomy_counts, left_on='taxon_id', right_on='NCBI Taxonomy ID', how='left')
abundance.set_index('taxon_id', inplace=True)
meta_cols = [
    'id', 'NCBI Taxonomy ID', 'kingdom', 'phylum', 'class', 'order',
    'family', 'genus', 'species', 'RefSeq', 'organism_name', 'genome_size_ungapped'
]

abundance['genome_size_ungapped'] = pd.to_numeric(abundance['genome_size_ungapped'], errors='coerce')
data_cols = abundance.columns.difference(meta_cols)
abundance[data_cols] = abundance[data_cols].div(abundance['genome_size_ungapped'], axis=0)
abundance = abundance.drop(columns=['RefSeq', 'organism_name', 'genome_size_ungapped', 'NCBI Taxonomy ID']).reset_index()
abundance.drop(columns=['kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species', 'id']).set_index('taxon_id').to_csv('./processed/abundance_normalized_table.csv')
abundance = abundance.drop(columns=['id']).melt(id_vars=['taxon_id', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species'], var_name='sample_id', value_name='abundance').rename(columns={'taxon_id': 'id'})
abundance['sample_id'] = abundance['sample_id'].astype(str)
abundance.to_csv('./processed/abundance_normalized.csv')
abundance

,id,kingdom,phylum,class,order,family,genus,species,sample_id,abundance
0,465515,Bacteria,Actinobacteria,Actinobacteria,Actinomycetales,Micrococcaceae,Micrococcus,Micrococcus luteus,33-0,2.444211e-05
1,469602,Bacteria,Fusobacteria,Fusobacteriia,Fusobacteriales,Fusobacteriaceae,Fusobacterium,Fusobacterium nucleatum,33-0,4.565735e-07
2,1002364,Bacteria,Proteobacteria,Gammaproteobacteria,Enterobacteriales,Enterobacteriaceae,Hafnia,Hafnia alvei,33-0,4.097535e-07
3,469599,Bacteria,Fusobacteria,Fusobacteriia,Fusobacteriales,Fusobacteriaceae,Fusobacterium,Fusobacterium periodonticum,33-0,2.830734e-06
4,502347,Bacteria,Proteobacteria,Gammaproteobacteria,Enterobacteriales,Enterobacteriaceae,Escherichia,Escherichia albertii,33-0,6.384972e-07
...,...,...,...,...,...,...,...,...,...,...
54803,520999,Bacteria,Proteobacteria,Gammaproteobacteria,Enterobacteriales,Enterobacteriaceae,Providencia,Providencia alcalifaciens,36-0,1.489075e-06
54804,1237085,Archaea,Thaumarchaeota,Nitrososphaeria,Nitrososphaerales,Nitrososphaeraceae,Nitrososphaera,Candidatus Nitrososphaera,36-0,0.000000e+00
54805,1217689,Bacteria,Proteobacteria,Gammaproteobacteria,Pseudomonadales,Moraxellaceae,Acinetobacter,Acinetobacter pittii,36-0,3.541527e-06
54806,1112856,Bacteria,Tenericutes,Mollicutes,Mycoplasmatales,Mycoplasmataceae,Mycoplasma,Mycoplasma pneumoniae,36-0,1.223727e-06


In [3]:
metadata = build(abundance, 
                 model_db='./raw/agora_species_103.qza',
                 out_folder='./models',
                 cutoff=0.0001,
                 threads=12)

Output()

In [4]:
western = load_qiime_medium('./raw/western_diet_gut_agora.qza')
res = grow(manifest=metadata,
           medium=western,
           model_folder='./models',
           threads=12,
           tradeoff=0.7)
res.save('./models/res_western.zip')

Output()

In [13]:
res = load_results('./models/res_western.zip')
growth_rates = res.growth_rates.copy()
growth_rates.taxon = growth_rates.taxon.astype(str)
growth_rates.sample_id = growth_rates.sample_id.astype(str)
exchanges = res.exchanges.copy()
exchanges.taxon = exchanges.taxon.astype(str)
annotations = res.annotations.copy()
annotations.metabolite = annotations.metabolite.astype(str)
res = GrowthResults(growth_rates=growth_rates, exchanges=exchanges, annotations=annotations)
interactions_res = interactions(results=res, threads=12, taxa=None)
interactions_res.to_csv('./models/interactions_western.csv')    

Output()

In [9]:
abundance_simulated = pd.read_csv('./raw/SimulatedFMTCounts.csv')
taxonomy = pd.read_csv('./processed/full_taxonomy.csv')

abundance_simulated = (
    abundance_simulated
    .merge(
        taxonomy, 
        left_on='taxon', 
        right_on='taxon_id', 
        how='left'
    )
    .drop(columns=['strain'])
    .melt(
        id_vars=['taxon_id', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species'],
        var_name='sample_id',
        value_name='abundance'
    )
    .sort_values(by='abundance', ascending=False)
    .groupby(
        ['sample_id', 'species'],
        as_index=False
    )
    .agg(
        {
            'abundance': 'sum',
            **{col: 'first' for col in ['taxon_id', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']}
        }
    )
    .assign(taxon_id=lambda x: x['taxon_id'].astype(str))
    .rename(columns={'taxon_id': 'id'})
)
abundance_simulated

,sample_id,abundance,id,kingdom,phylum,class,order,family,genus,species
0,D-10_vs_11-0,79729,679935,Bacteria,Bacteroidetes,Bacteroidia,Bacteroidales,Rikenellaceae,Alistipes,Alistipes finegoldii
1,D-10_vs_11-0,2434,742725,Bacteria,Bacteroidetes,Bacteroidia,Bacteroidales,Rikenellaceae,Alistipes,Alistipes indistinctus
2,D-10_vs_11-0,169014,1203611,Bacteria,Bacteroidetes,Bacteroidia,Bacteroidales,Rikenellaceae,Alistipes,Alistipes onderdonkii
3,D-10_vs_11-0,177788,445970,Bacteria,Bacteroidetes,Bacteroidia,Bacteroidales,Rikenellaceae,Alistipes,Alistipes putredinis
4,D-10_vs_11-0,77379,717959,Bacteria,Bacteroidetes,Bacteroidia,Bacteroidales,Rikenellaceae,Alistipes,Alistipes shahii
...,...,...,...,...,...,...,...,...,...,...
52320,taxon,742821,742821,Bacteria,Proteobacteria,Betaproteobacteria,Burkholderiales,Sutterellaceae,Sutterella,Sutterella wadsworthensis
52321,taxon,866776,866776,Bacteria,Firmicutes,Negativicutes,Veillonellales,Veillonellaceae,Veillonella,Veillonella atypica
52322,taxon,546273,546273,Bacteria,Firmicutes,Negativicutes,Veillonellales,Veillonellaceae,Veillonella,Veillonella dispar
52323,taxon,479436,479436,Bacteria,Firmicutes,Negativicutes,Selenomonadales,Veillonellaceae,Veillonella,Veillonella parvula


In [ ]:
metadata_simulated = build(abundance_simulated,
                           model_db='./raw/agora_species_103.qza',
                           out_folder='./models_simulated',
                           cutoff=0.0001,
                           threads=12)